# Phase 4, Stage 3c: Pairwise t-tests and Cohen's d

## What this adds

The KS test (3a) and chi-squared test (3b) both look at *distributional shape*. This step asks the more conventional question: **does the *average* CPL differ between adjacent time pressure bins?**

- **H0:** the mean `capped_cpl` is the same in both bins being compared.
- **H1:** the mean `capped_cpl` differs between the two bins.

Same structure as the KS tests: adjacent bins (1 vs 2, 2 vs 3, 3 vs 4) within each rating band = **15 comparisons**. For consistency with the KS tests (also 15 comparisons), we apply the same Bonferroni-corrected **α = 0.0033**.

## Which t-test, and why

A standard (Student's) t-test assumes the two groups have equal variance. From Stage 1, we already know the standard deviation of `capped_cpl` changes substantially across bins within a band (e.g. for Band 1, std ranges from ~100 to ~111). That assumption doesn't hold here, so we use **Welch's t-test** (`equal_var=False`), which doesn't assume equal variances and is generally considered the safer default.

## Cohen's d

Cohen's d expresses the difference in means in terms of **standard deviations** — "how many SDs apart are these two group means?" It's calculated as:

$$d = \\frac{\\bar{x}_1 - \\bar{x}_2}{s_{pooled}}$$

where $s_{pooled}$ is the pooled standard deviation across both groups. Conventional (rough) benchmarks: **0.2 = small, 0.5 = medium, 0.8 = large**. These benchmarks were developed for psychology research generally, not chess/CPL specifically — worth keeping in mind when interpreting.

As before: **failing to reject H0, or finding a small Cohen's d, is a legitimate and informative result** — it would tell us that even though the *shape* of the distribution may shift (per the KS results), the *average* move quality doesn't change much. That's actually consistent with the Stage 1 finding that medians often barely move while the tail does the work.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

ALPHA_BONFERRONI = 0.05 / 15  # 0.0033, consistent with the KS tests

RATING_BAND_LABELS = {
    1: 'Novice (<1000)',
    2: 'Intermediate (1000-1499)',
    3: 'Club Player (1500-1999)',
    4: 'Advanced (2000-2299)',
    5: 'Expert/Master (2300+)',
}

TIME_PRESSURE_LABELS = {
    1: 'Minimal (>75%)',
    2: 'Low (50-75%)',
    3: 'Moderate (25-50%)',
    4: 'High (<25%)',
}

df = pd.read_csv('../../data/processed/analysed_moves.csv')
print(f'Loaded {len(df):,} rows')
print(f'Bonferroni-corrected alpha for t-tests: {ALPHA_BONFERRONI:.4f}')

In [ ]:
def cohens_d(sample_a, sample_b):
    n_a, n_b = len(sample_a), len(sample_b)
    var_a, var_b = sample_a.var(ddof=1), sample_b.var(ddof=1)
    pooled_sd = np.sqrt(((n_a - 1) * var_a + (n_b - 1) * var_b) / (n_a + n_b - 2))
    return (sample_a.mean() - sample_b.mean()) / pooled_sd

adjacent_transitions = [(1, 2), (2, 3), (3, 4)]

results = []
for rating_band in sorted(RATING_BAND_LABELS):
    for bin_a, bin_b in adjacent_transitions:
        sample_a = df.loc[(df['rating_band'] == rating_band) & (df['time_pressure_bin'] == bin_a), 'capped_cpl']
        sample_b = df.loc[(df['rating_band'] == rating_band) & (df['time_pressure_bin'] == bin_b), 'capped_cpl']

        t_result = stats.ttest_ind(sample_a, sample_b, equal_var=False)
        d = cohens_d(sample_a, sample_b)

        results.append({
            'rating_band': rating_band,
            'rating_band_label': RATING_BAND_LABELS[rating_band],
            'transition': f'Bin {bin_a} -> Bin {bin_b}',
            'mean_a': sample_a.mean(),
            'mean_b': sample_b.mean(),
            'mean_diff': sample_a.mean() - sample_b.mean(),
            't_statistic': t_result.statistic,
            'p_value': t_result.pvalue,
            'significant_bonferroni': t_result.pvalue < ALPHA_BONFERRONI,
            'cohens_d': d,
        })

ttest_results = pd.DataFrame(results)
ttest_results

## Save results and view Cohen's d by band/transition

Same pivot structure as the KS D-statistic table, for direct side-by-side comparison later.

In [ ]:
ttest_results.to_csv('../results/ttest_cohens_d_results.csv', index=False)
print('Saved to ../results/ttest_cohens_d_results.csv')

d_pivot = ttest_results.pivot(index='rating_band_label', columns='transition', values='cohens_d')
d_pivot = d_pivot.reindex(index=[RATING_BAND_LABELS[b] for b in sorted(RATING_BAND_LABELS)],
                           columns=['Bin 1 -> Bin 2', 'Bin 2 -> Bin 3', 'Bin 3 -> Bin 4'])
d_pivot